# 1. Compute Security — Bastion, JIT, AKS, Containers, Disk Encryption

Welcome! This notebook is for someone new to Azure security. We will learn how to
**protect the machines that run your code** in Azure — virtual machines (VMs),
Kubernetes clusters (AKS), and containers.

### What you'll learn
1. **Azure Bastion** — connect to VMs without giving them a public IP.
2. **Just-In-Time (JIT) VM Access** — open management ports only when needed.
3. **AKS security** — private clusters, workload identity, network policy.
4. **Container registry (ACR)** — sign images, scan for vulnerabilities.
5. **Disk encryption** — SSE, ADE, encryption at host, confidential disks.
6. **Trusted Launch & Confidential VMs** — protect the VM itself, not just disks.

> 🎯 **Exam weight**: ~20–25 % of AZ-500. Expect questions that combine several
> of these features (e.g., "secure an AKS cluster that must pull images from ACR
> and authenticate to a database").

### Analogy 🏢
Think of an Azure VM like an office in a building:
- **Public IP + open RDP/SSH** = leaving the door on the street unlocked.
- **Bastion** = there is no street door; you enter via a guarded lobby.
- **JIT** = the door exists, but the guard only unlocks it when you ring the bell.
- **Disk encryption** = the filing cabinets inside are locked even if someone
  steals the whole building.

## Before you run this notebook

1. From the lab folder (`security-certs/az-500/03-compute-storage-databases`) run `uv sync`.
2. In VS Code, click the **kernel picker** (top-right of this notebook) and pick
   the interpreter from `.venv`.
3. If the kernel isn't listed, reload the window (`Cmd+Shift+P` → *Reload Window*).

No Azure subscription needed — everything is simulated in Python so you can
learn the concepts safely.

## 1. Azure Bastion — connecting to VMs without a public IP

**Problem (bad practice)**: VMs with a public IP and open port 22/3389.
Internet-facing SSH is probed by bots within seconds of going live.

**Solution (best practice)**: deploy an **Azure Bastion** host inside your VNet.
Users open the Azure portal, and Bastion proxies RDP/SSH through HTTPS (port 443).
The VM itself only needs a **private** IP.

```bash
# 1. Create the dedicated subnet (MUST be named exactly "AzureBastionSubnet", /26 or bigger)
az network vnet subnet create -g rg-prod --vnet-name vnet-prod \
  -n AzureBastionSubnet --address-prefixes 10.0.3.0/26

# 2. Public IP for the Bastion host itself (the VMs stay private)
az network public-ip create -g rg-prod -n bastion-pip \
  --sku Standard --allocation-method Static

# 3. Create the Bastion host
az network bastion create -g rg-prod -n bastion-prod \
  --vnet-name vnet-prod --public-ip-address bastion-pip \
  --sku Standard        # Standard enables file transfer & peered-VNet access
```

### Bastion SKUs at a glance

| Feature                               | Developer | Basic | Standard | Premium |
|---------------------------------------|:---------:|:----:|:--------:|:-------:|
| Needs an `AzureBastionSubnet`         |  ❌ (shared, free) |  ✅  |    ✅    |   ✅    |
| RDP/SSH to VMs in same VNet           |  ✅  |  ✅  |    ✅    |   ✅    |
| RDP/SSH to VMs in peered VNets        |  ❌  |  ❌  |    ✅    |   ✅    |
| Connect by IP address                 |  ❌  |  ❌  |    ✅    |   ✅    |
| File upload / download                |  ❌  |  ❌  |    ✅    |   ✅    |
| Shareable link (no Azure login)       |  ❌  |  ❌  |    ✅    |   ✅    |
| Host scaling (extra instances)        |  ❌  |  ❌  |    ✅    |   ✅    |
| Private-only Bastion (no public IP)   |  ❌  |  ❌  |    ❌    |   ✅    |
| Session recording                     |  ❌  |  ❌  |    ❌    |   ✅    |

> **Exam tips**
> - The subnet MUST be named exactly `AzureBastionSubnet` and be **/26 or larger**.
>   You cannot put other resources in it, and it must not have a route table that
>   forces Bastion's own traffic elsewhere.
> - Bastion needs a **Standard SKU, static** public IP (except the Developer SKU,
>   which has no subnet and no public IP, and the Premium private-only deployment).
> - The target VM needs **no** public IP, **no** agent, and **no** inbound rule from
>   the internet — only an inbound Allow for 3389/22 from the `AzureBastionSubnet`
>   range (or the `VirtualNetwork` service tag).

In [ ]:
# Bastion connectivity simulator — who can reach which VM?
VMS = {
    'web-vm':   {'public_ip': None,      'private_ip': '10.0.1.10', 'vnet': 'vnet-prod'},
    'legacy-vm':{'public_ip': '20.1.2.3','private_ip': '10.0.1.11', 'vnet': 'vnet-prod'},
    'db-vm':    {'public_ip': None,      'private_ip': '10.0.2.10', 'vnet': 'vnet-peered'},
}
BASTION = {'name': 'bastion-prod', 'sku': 'Standard', 'vnet': 'vnet-prod',
           'peered_vnets': ['vnet-peered']}

def connect_via_bastion(user_ip, target):
    vm = VMS[target]
    if BASTION['sku'] == 'Basic' and vm['vnet'] != BASTION['vnet']:
        return f"❌ Basic SKU can't reach peered VNet '{vm['vnet']}'. Upgrade to Standard."
    if vm['vnet'] != BASTION['vnet'] and vm['vnet'] not in BASTION['peered_vnets']:
        return f"❌ VNet '{vm['vnet']}' is not peered with Bastion's VNet."
    return (f"✅ User {user_ip} → Azure Portal (HTTPS) → Bastion → {target} "
            f"at {vm['private_ip']}  (no public IP needed on the VM)")

def connect_directly(user_ip, target):
    vm = VMS[target]
    if vm['public_ip'] is None:
        return f"❌ {target} has no public IP. Direct SSH/RDP impossible."
    return (f"⚠️ User {user_ip} → Internet → {vm['public_ip']}:22  "
            f"(VM exposed to every port-scanner on the planet)")

print('=== Scenario A — connect to a VM that has NO public IP ===')
print('Direct:   ', connect_directly('203.0.113.50', 'web-vm'))
print('Bastion:  ', connect_via_bastion('203.0.113.50', 'web-vm'))

print('\n=== Scenario B — legacy VM that still has a public IP ===')
print('Direct:   ', connect_directly('203.0.113.50', 'legacy-vm'))
print('💡 Remove the public IP once Bastion is in place.')

print('\n=== Scenario C — VM in a peered VNet ===')
print('Bastion:  ', connect_via_bastion('203.0.113.50', 'db-vm'))

## 2. Just-In-Time (JIT) VM Access

Sometimes you **must** keep ports 22/3389 reachable (legacy tools that can't
use Bastion). JIT keeps those ports **closed by default** and opens them for a
specific user, a specific IP, and a specific time window — then slams them shut
again.

JIT requires **Microsoft Defender for Servers** (a paid Defender for Cloud plan).

```bash
# Enable JIT on a VM (declaratively — a policy object in Defender for Cloud)
az rest --method PUT \
  --uri 'https://management.azure.com/subscriptions/{sub}/resourceGroups/rg-prod/providers/Microsoft.Security/locations/eastus/jitNetworkAccessPolicies/default?api-version=2020-01-01' \
  --body @jit-policy.json

# When an admin needs to connect they "request access". There is no stable
# `az security jit-policy` verb for this -- the portal (and the SDKs) call the
# `initiate` action on the policy directly:
az rest --method POST \
  --uri 'https://management.azure.com/subscriptions/{sub}/resourceGroups/rg-prod/providers/Microsoft.Security/locations/eastus/jitNetworkAccessPolicies/default/initiate?api-version=2020-01-01' \
  --body '{"virtualMachines":[{"id":"/subscriptions/.../virtualMachines/my-vm",
           "ports":[{"number":22,"duration":"PT1H","allowedSourceAddressPrefix":"203.0.113.50"}]}]}'

# `az security jit-policy list / show` do exist and are useful for auditing:
az security jit-policy list -o table
```

### What JIT actually does under the hood

When the request is approved, Defender for Cloud **adds a temporary Allow rule to
the NSG (and Azure Firewall, if the VM sits behind one)** with a priority *lower*
(numerically) than the deny rule that blocks the port, scoped to the requester's
source IP. When the window expires it **removes that rule again**. It does not
touch the VM. That is why:

- JIT protects VMs that still have a **public IP** (or are behind Azure Firewall).
- JIT is **not** an alternative to Bastion for eliminating public IPs — it reduces
  the *time window*, not the *attack surface shape*.
- Max request duration is capped by the policy (Azure allows up to **24 hours**).

### Bastion vs JIT — when to use which

|                                | Bastion | JIT |
|--------------------------------|:------:|:---:|
| VM keeps a public IP?          |   ❌   |  ✅ (just ports closed) |
| Requires agent on the VM       |   ❌   |  ❌ |
| Protocol                       | RDP/SSH via browser | Native clients |
| Good for                       | Default choice | Legacy/native-client scenarios |
| Cost                           | Bastion host hourly | Defender for Servers plan |

In [ ]:
# Simulate a full JIT request lifecycle
from datetime import datetime, timedelta
import json

class JITPolicy:
    def __init__(self):
        self.vms = {}                # vm -> {ports:[...], active:[...]}
        self.nsg_rules = []          # timeline of rule adds/removes
    def enroll(self, vm, ports, max_hours=3):
        self.vms[vm] = {'ports': [{'number': p, 'max_hours': max_hours} for p in ports],
                        'active': []}
    def request(self, vm, port, source_ip, hours, user):
        if vm not in self.vms:
            return {'status': '❌ VM not enrolled in JIT'}
        cfg = next((p for p in self.vms[vm]['ports'] if p['number'] == port), None)
        if not cfg:
            return {'status': f'❌ Port {port} not in JIT policy'}
        if hours > cfg['max_hours']:
            return {'status': f'❌ Requested {hours}h exceeds max {cfg["max_hours"]}h'}
        now = datetime(2025, 1, 15, 10, 0)   # fixed clock for reproducible output
        expires = now + timedelta(hours=hours)
        rule = {'name': f'JIT-{user}-{port}-{source_ip}',
                'priority': 100, 'action': 'Allow',
                'src': source_ip, 'dst_port': port,
                'added': now.strftime('%H:%M'),
                'removed_at': expires.strftime('%H:%M')}
        self.nsg_rules.append(('ADD', rule))
        self.vms[vm]['active'].append(rule)
        return {'status': '✅ Access granted', 'rule': rule,
                'note': 'NSG ALLOW rule auto-removed at removed_at.'}
    def cleanup_expired(self, at_time):
        for vm, cfg in self.vms.items():
            still_active = []
            for rule in cfg['active']:
                if rule['removed_at'] <= at_time.strftime('%H:%M'):
                    self.nsg_rules.append(('REMOVE', rule))
                else:
                    still_active.append(rule)
            cfg['active'] = still_active

jit = JITPolicy()
jit.enroll('finance-vm', ports=[22, 3389], max_hours=3)

print('--- Admin Alice requests 1h of SSH from her laptop ---')
print(json.dumps(jit.request('finance-vm', 22, '203.0.113.50', 1, 'alice'), indent=2, default=str))

print('\n--- Attacker Mallory tries to request 24h for "*" ---')
print(json.dumps(jit.request('finance-vm', 22, '0.0.0.0/0', 24, 'mallory'), indent=2, default=str))

print('\n--- 61 minutes later, Defender auto-cleans the NSG rule ---')
jit.cleanup_expired(datetime(2025, 1, 15, 11, 1))
for event, rule in jit.nsg_rules:
    print(f'  {event:6s} {rule["name"]}  (was open {rule["added"]}→{rule["removed_at"]})')

## 3. AKS Security — six levers you must know

AKS (Azure Kubernetes Service) is the most-asked compute topic on AZ-500.
Always remember these six switches:

| Lever | CLI flag | Why it matters |
|-------|----------|---------------|
| **Private cluster**        | `--enable-private-cluster` | API server gets a private IP (no Internet exposure) |
| **Entra ID integration**   | `--enable-aad` `--aad-admin-group-object-ids <id>` | RBAC maps to Entra groups; no local `kubeconfig` secrets |
| **Workload identity**      | `--enable-oidc-issuer --enable-workload-identity` | Pods get federated managed-identity tokens (no client secrets) |
| **Network policy**         | `--network-policy calico` (or `azure`) | East-west firewall between pods |
| **Azure Policy add-on**    | `--enable-addons azure-policy` | OPA Gatekeeper rules enforce pod security baseline |
| **ACR attach**             | `--attach-acr <acr>` | AKS kubelet pulls images via managed identity — no registry credentials in YAML |

```bash
# Bad practice — public API server, SQL auth, no network policy, keys in secrets
az aks create -g rg-prod -n aks-prod --generate-ssh-keys   # everything default ❌

# Best practice — hardened cluster
az aks create -g rg-prod -n aks-prod \
  --enable-private-cluster \
  --enable-aad --aad-admin-group-object-ids <sec-group-id> \
  --enable-oidc-issuer --enable-workload-identity \
  --network-policy calico \
  --enable-addons azure-policy \
  --attach-acr myacr \
  --disable-local-accounts         # force Entra auth only
```

### Workload identity flow (no secrets in pods)

```
┌──────────────┐     ┌──────────────┐    ┌────────────────┐
│ Pod's SA token│ →  │  Entra ID    │ →  │  Access token  │
│  (projected)  │    │ (federated)  │    │  for Key Vault │
└──────────────┘     └──────────────┘    └────────────────┘
```
1. Kubernetes projects a **service-account token** into the pod.
2. The pod's SDK exchanges that token for an **Entra access token** via the
   cluster's OIDC issuer.
3. The Entra token is used to call Azure resources (Key Vault, SQL, Storage).

> 🏆 Result: zero client secrets in your cluster. Compromising a pod no longer
> compromises a long-lived credential.

In [ ]:
# AKS security self-assessment — score your cluster config (bad → best)
CONFIGS = {
    'prod-v1 (bad)': {
        'private_cluster': False, 'aad_integration': False,
        'workload_identity': False, 'network_policy': None,
        'azure_policy_addon': False, 'attached_acr': False,
        'local_accounts_disabled': False,
    },
    'prod-v2 (better)': {
        'private_cluster': True, 'aad_integration': True,
        'workload_identity': False, 'network_policy': 'azure',
        'azure_policy_addon': False, 'attached_acr': True,
        'local_accounts_disabled': False,
    },
    'prod-v3 (best)': {
        'private_cluster': True, 'aad_integration': True,
        'workload_identity': True, 'network_policy': 'calico',
        'azure_policy_addon': True, 'attached_acr': True,
        'local_accounts_disabled': True,
    },
}

CHECKS = [
    ('private_cluster',        lambda v: v is True,        'Private API server'),
    ('aad_integration',        lambda v: v is True,        'Entra ID auth'),
    ('workload_identity',      lambda v: v is True,        'Workload identity (no pod secrets)'),
    ('network_policy',         lambda v: v in ('azure','calico'), 'Pod-to-pod firewall'),
    ('azure_policy_addon',     lambda v: v is True,        'Azure Policy add-on'),
    ('attached_acr',           lambda v: v is True,        'ACR attached (managed-identity pulls)'),
    ('local_accounts_disabled',lambda v: v is True,        'No local accounts (Entra only)'),
]

for name, cfg in CONFIGS.items():
    passes = sum(1 for k, ok, _ in CHECKS if ok(cfg.get(k)))
    bar = '█' * passes + '░' * (len(CHECKS) - passes)
    print(f'\n{name}  [{bar}]  {passes}/{len(CHECKS)}')
    for k, ok, desc in CHECKS:
        mark = '✅' if ok(cfg.get(k)) else '❌'
        print(f'  {mark} {desc}')

## 4. Container Registry (ACR) security

Your images are **executable code**. Treat ACR with the same paranoia as your
source repo.

| Feature | Why | CLI |
|---------|-----|-----|
| **Private endpoint** | Registry only reachable from your VNet | `az acr update -n myacr --public-network-enabled false` + private endpoint |
| **Content trust (signing)** | Consumers refuse unsigned images | `az acr config content-trust update -n myacr --status enabled` (Premium SKU) |
| **Vulnerability scanning** | Find known CVEs in layers | Enable **Defender for Containers** |
| **Retention & purge** | Remove old, unused tags | `az acr config retention update -n myacr --status enabled --days 30 --type UntaggedManifests` |
| **Token / scope-map** | Fine-grained creds per repo | `az acr token create ... --scope-map <map>` |
| **Managed-identity pull** | No admin user, no secrets | `az acr update -n myacr --admin-enabled false` |

In [ ]:
# Simulate an admission controller that enforces:
#  (a) only images from our trusted ACR
#  (b) image must be signed
#  (c) no CRITICAL vulnerabilities from last scan
TRUSTED_REGISTRY = 'myacr.azurecr.io'
SCAN_RESULTS = {
    'myacr.azurecr.io/api:v1.2.0':   {'signed': True,  'critical': 0, 'high': 2},
    'myacr.azurecr.io/api:v1.1.0':   {'signed': True,  'critical': 3, 'high': 5},
    'myacr.azurecr.io/api:dev':      {'signed': False, 'critical': 0, 'high': 1},
    'docker.io/library/nginx:latest':{'signed': False, 'critical': 1, 'high': 7},
}

def admit(pod_image):
    registry = pod_image.split('/')[0]
    if registry != TRUSTED_REGISTRY:
        return f'❌ DENY  {pod_image}  (registry not in allowlist)'
    scan = SCAN_RESULTS.get(pod_image)
    if scan is None:
        return f'❌ DENY  {pod_image}  (no recent vulnerability scan)'
    if not scan['signed']:
        return f'❌ DENY  {pod_image}  (image is not signed / content trust)'
    if scan['critical'] > 0:
        return f'❌ DENY  {pod_image}  ({scan["critical"]} CRITICAL CVEs)'
    return f'✅ ADMIT {pod_image}  ({scan["high"]} HIGH CVEs — monitor)'

for img in ['myacr.azurecr.io/api:v1.2.0', 'myacr.azurecr.io/api:v1.1.0',
            'myacr.azurecr.io/api:dev',    'docker.io/library/nginx:latest']:
    print(admit(img))

## 5. Disk encryption options — which one for which threat?

| Method | What it encrypts | Key location | Protects against |
|--------|-----------------|--------------|------------------|
| **SSE (default, always on)** | Managed disks at rest | Platform-managed key (PMK) | Stolen physical disk |
| **SSE with CMK** | Same + you control the key | Your Key Vault / Managed HSM | Same + gives you kill-switch |
| **ADE (Azure Disk Encryption)** | OS + data volumes (BitLocker / dm-crypt) | Key Vault | Offline VHD theft, boot-disk access |
| **Encryption at host** | OS + data disks *and* temp disk, host cache | PMK or CMK | Caches/temp that SSE misses |
| **Confidential disk** | Disk tied to a Trusted Launch VM; key released only inside the TEE | Managed HSM | Compromised host / hypervisor |

**Rule of thumb for the exam:**
- "Encrypt temp disks and caches too" → **Encryption at host**.
- "Encrypt even from the host / hypervisor" → **Confidential disk (+ Confidential VM)**.
- "BitLocker-level guest OS encryption" → **ADE**.
- "Customer-controlled key, simplest" → **SSE with CMK**.

### The three gotchas the exam loves

1. **ADE and encryption-at-host are mutually exclusive.** You cannot enable both on
   the same VM. Pick one: ADE if you need in-guest BitLocker/dm-crypt semantics,
   encryption at host if you want the platform to handle it (and to cover the
   **temp disk and host cache**, which SSE alone does not).
2. **ADE requires a Key Vault in the same region and subscription as the VM, with
   the vault's `enabledForDiskEncryption` (Azure Disk Encryption) access setting
   turned on.** Miss that and the extension fails.
3. **SSE is always on and cannot be disabled.** A question that says "encryption is
   turned off for managed disks" is describing something impossible — the real
   answer is usually "switch the key from platform-managed to customer-managed".

```bash
# Key Vault must be enabled for disk encryption BEFORE running az vm encryption enable
az keyvault update -n my-kv -g rg-prod --enabled-for-disk-encryption true
```

```bash
# Enable ADE on a running VM
az vm encryption enable -g rg-prod -n my-vm \
  --disk-encryption-keyvault /subscriptions/.../vaults/my-kv \
  --volume-type All

# Or — encryption at host (simpler, no extra tooling)
az vm update -g rg-prod -n my-vm --set securityProfile.encryptionAtHost=true
# (Feature must be registered on the subscription first.)
```

## 6. Trusted Launch & Confidential VMs (don't forget these!)

Beyond disks, Azure can protect the **VM itself**:

| Feature | Protects against | How to enable |
|---------|-----------------|---------------|
| **Secure Boot**            | Rootkits / unsigned bootloaders | `--security-type TrustedLaunch --enable-secure-boot true` |
| **vTPM**                   | Tamper-proof attestation        | `--enable-vtpm true` |
| **Boot integrity monitoring** | Detect unexpected boot changes | Defender for Servers |
| **Confidential VM (AMD SEV-SNP)** | Hypervisor / Azure operators | `--security-type ConfidentialVM` + supported SKU (DCasv5 / ECasv5) |

---
## Summary

| Feature | Key implementation detail |
|---------|--------------------------|
| **Bastion**          | `AzureBastionSubnet` (/26+), Standard SKU for peered VNets & file transfer |
| **JIT**              | Needs Defender for Servers. NSG rules are **temporary** and auto-removed. |
| **AKS**              | Private cluster + Entra + workload identity + network policy + Azure Policy + ACR attach |
| **ACR**              | Private endpoint, admin user OFF, content trust ON, Defender scanning |
| **Disk encryption**  | SSE (on) + ADE (guest OS) + encryption-at-host (caches) + Confidential disk (TEE) |
| **Trusted Launch**   | Secure Boot + vTPM. Confidential VM extends protection against the host. |

**Next**: [Notebook 2 — Storage Security](02_storage_security.ipynb)

---
## ✅ Self-check

1. A VM has no public IP. Can JIT VM access help? Can Bastion?
2. Which Bastion SKU do you need to reach a VM in a **peered** VNet, and which do you need
   to deploy Bastion with **no public IP at all**?
3. You must encrypt the VM's **temp disk** and the **host cache** as well as the OS and data
   disks, with a key you control. Which feature, and what can you *not* also enable?
4. A regulator demands that neither Microsoft operators nor the hypervisor can read the VM's
   memory. What do you deploy?
5. Your AKS cluster pulls images from ACR. How do you do it with **no** registry credentials
   stored anywhere?
6. Which two AKS flags together give pods an Entra identity without any client secret?

In [ ]:
answers = """
1. JIT: only partially -- JIT works by opening/closing NSG (and Azure Firewall) rules,
   so it is aimed at VMs that ARE reachable (public IP, or DNAT through a firewall).
   With no public IP and no DNAT there is nothing for JIT to gate.
   BASTION: yes -- that is exactly its use case. Bastion brokers RDP/SSH over HTTPS
   from the portal to the VM's PRIVATE IP.

2. Peered VNets -> STANDARD (Basic is same-VNet only).
   Bastion with no public IP (private-only deployment) -> PREMIUM.

3. ENCRYPTION AT HOST with a customer-managed key (a disk encryption set backed by
   your Key Vault / Managed HSM). You then CANNOT also enable Azure Disk Encryption
   -- ADE and encryption at host are mutually exclusive on the same VM.
   Also note encryption at host must be registered as a subscription feature and is
   not supported on every VM size.

4. A CONFIDENTIAL VM (AMD SEV-SNP, e.g. DCasv5 / ECasv5) with confidential OS disk
   encryption. Trusted Launch (Secure Boot + vTPM) protects the BOOT chain but does
   NOT hide memory from the host -- that distinction is a favourite exam question.

5. Attach the registry to the cluster: `az aks update --attach-acr <acr>`. This grants
   the cluster's kubelet MANAGED IDENTITY the AcrPull role on the registry, so image
   pulls are authorised by Entra ID. No imagePullSecret, and you can leave the ACR
   admin user disabled.

6. --enable-oidc-issuer and --enable-workload-identity. Together they let Kubernetes
   project a service-account token that Entra ID trusts via federated credentials,
   which the pod exchanges for an Entra access token. No client secret ever exists.
"""
print(answers)